# 05 — GPU Performance

## Objective

This notebook studies how the **same GELU-style computation** behaves across different execution strategies, from a pure Python CPU loop to optimized and custom GPU kernels.

The implementations explored are:

1. Pure Python on CPU
2. Custom PyTorch operations on GPU
3. PyTorch with `torch.compile`
4. PyTorch built-in GELU
5. Custom CUDA C++ kernel
6. Triton kernel

The focus is not only on *which implementation is faster*, but on **why** performance changes: parallelism, kernel-launch overhead, kernel fusion, memory traffic, workload size, and implementation quality.

> **Benchmark note:** the recorded built-in PyTorch timing uses default `F.gelu(x)`, while the custom implementations use the tanh approximation. The numerical difference is small, but the final notes flag this because tiny timing gaps between optimized implementations should not be overinterpreted. A strictly matched `approximate="tanh"` benchmark cell is included later for rerunning on a GPU.


## Environment

The experiments were run in Google Colab with a CUDA-enabled **Tesla T4** GPU.


In [1]:
import math
import os
import random
import time

import torch
import torch.nn as nn
import torch.nn.functional as F

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")


CUDA available: True
GPU: Tesla T4


---

## Level 1 — Pure Python CPU Baseline

The baseline evaluates the tanh approximation of GELU one scalar at a time using ordinary Python. This intentionally exposes Python-loop overhead and provides a simple reference point rather than an optimized CPU implementation.


In [ ]:
def gelu_python(x):
    return 0.5 * x * (
        1 + math.tanh(
            math.sqrt(2.0 / math.pi) *
            (x + 0.044715 * (x ** 3))
        )
    )


In [2]:
print(gelu_python(1.0))

0.8411919906082768


In [ ]:
def benchmark_python(n):
    random.seed(42)
    x = [random.uniform(-3, 3) for _ in range(n)]

    start = time.perf_counter()
    _ = [gelu_python(v) for v in x]
    end = time.perf_counter()

    return end - start


In [3]:
sizes = [32_000, 256_000, 1_000_000, 8_000_000]

for n in sizes:
    t = benchmark_python(n)
    print(f"{n:,} elements -> {t:.6f} seconds")

32,000 elements -> 0.010096 seconds
256,000 elements -> 0.076475 seconds
1,000,000 elements -> 0.424910 seconds
8,000,000 elements -> 4.444750 seconds


---

## Level 2 — Custom PyTorch Operations on GPU

The same equation is expressed with PyTorch tensor operations. The tensor lives on the GPU, so the element-wise work can be executed in parallel, but the expression is still composed of several separate PyTorch operations.


In [ ]:
import torch
import time
import math

def gelu_torch(x):
    return 0.5 * x * (
        1 + torch.tanh(
            math.sqrt(2.0 / math.pi) *
            (x + 0.044715 * (x ** 3))
        )
    )

In [4]:
x = torch.tensor([1.0], device="cuda")

y = gelu_torch(x)

print(y)
print(x.device)

tensor([0.8412], device='cuda:0')
cuda:0


In [ ]:
def benchmark_torch_gpu(n, repeats=100):
    x = torch.empty(n, device="cuda").uniform_(-3, 3)

    # Warm-up
    for _ in range(10):
        _ = gelu_torch(x)

    torch.cuda.synchronize()

    start = time.perf_counter()

    for _ in range(repeats):
        y = gelu_torch(x)

    torch.cuda.synchronize()

    end = time.perf_counter()

    return (end - start) / repeats

In [5]:
sizes = [32_000, 256_000, 1_000_000, 8_000_000]

for n in sizes:
    t = benchmark_torch_gpu(n)
    print(f"{n:,} elements -> {t:.9f} seconds")

32,000 elements -> 0.000079650 seconds
256,000 elements -> 0.000085790 seconds
1,000,000 elements -> 0.000314404 seconds
8,000,000 elements -> 0.002415054 seconds


---

## Level 3 — `torch.compile`

`torch.compile` analyzes the computation graph and can fuse compatible element-wise operations into fewer GPU kernels. Compilation/warm-up is performed before steady-state timing so compilation overhead is not mixed with execution time.


In [ ]:
compiled_gelu = torch.compile(gelu_torch)

In [ ]:
x = torch.empty(1_000_000, device="cuda").uniform_(-3, 3)

for _ in range(10):
    _ = compiled_gelu(x)

torch.cuda.synchronize()

In [ ]:
def benchmark_compiled_gpu(n, repeats=100):
    x = torch.empty(n, device="cuda").uniform_(-3, 3)

    # Warm-up
    for _ in range(10):
        _ = compiled_gelu(x)

    torch.cuda.synchronize()

    start = time.perf_counter()

    for _ in range(repeats):
        y = compiled_gelu(x)

    torch.cuda.synchronize()

    end = time.perf_counter()

    return (end - start) / repeats

In [6]:
sizes = [32_000, 256_000, 1_000_000, 8_000_000]

for n in sizes:
    t = benchmark_compiled_gpu(n)
    print(f"{n:,} elements -> {t:.9f} seconds")

32,000 elements -> 0.000063203 seconds
256,000 elements -> 0.000062092 seconds
1,000,000 elements -> 0.000065964 seconds
8,000,000 elements -> 0.000266252 seconds


---

## Level 4 — PyTorch Built-in GELU

PyTorch includes an optimized built-in GELU implementation. The recorded benchmark below used the default `F.gelu(x)` mode. A tanh-matched version is also defined so correctness can be checked against the same mathematical approximation used by the custom kernels.


In [ ]:
def gelu_builtin(x):
    # Matches the recorded benchmark below (PyTorch default GELU mode).
    return F.gelu(x)

def gelu_builtin_tanh(x):
    # Strict mathematical match for the custom tanh-approximation kernels.
    return F.gelu(x, approximate="tanh")


In [ ]:
def benchmark_builtin_gpu(n, repeats=100):
    x = torch.empty(n, device="cuda").uniform_(-3, 3)

    # Warm-up
    for _ in range(10):
        _ = gelu_builtin(x)

    torch.cuda.synchronize()

    start = time.perf_counter()

    for _ in range(repeats):
        y = gelu_builtin(x)

    torch.cuda.synchronize()

    end = time.perf_counter()

    return (end - start) / repeats

In [7]:
sizes = [32_000, 256_000, 1_000_000, 8_000_000]

for n in sizes:
    t = benchmark_builtin_gpu(n)
    print(f"{n:,} elements -> {t:.9f} seconds")

32,000 elements -> 0.000008153 seconds
256,000 elements -> 0.000008282 seconds
1,000,000 elements -> 0.000036588 seconds
8,000,000 elements -> 0.000271121 seconds


---

## Level 5 — Custom CUDA C++ Kernel

This implementation moves to a lower level and explicitly maps GPU threads to tensor elements. The entire GELU approximation is computed inside one fused CUDA kernel.


In [ ]:
# Colab dependency for compiling the CUDA extension
!pip install ninja -q

from torch.utils.cpp_extension import load_inline
os.environ["NINJA_BUILD"] = "1"


In [ ]:
cuda_source = r"""
#include <torch/extension.h>
#include <cuda.h>
#include <cuda_runtime.h>

__global__ void fused_gelu_f32(float *out, const float *x, int n) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    if (idx < n) {
        float v = x[idx];

        float cube = v * v * v;

        float inner =
            sqrtf(2.0f / 3.14159265358979f) *
            (v + 0.044715f * cube);

        float t = tanhf(inner);

        out[idx] = 0.5f * v * (1.0f + t);
    }
}

torch::Tensor cuda_gelu(torch::Tensor x) {

    auto out = torch::empty_like(x);

    int n = x.numel();

    int threads = 256;

    int blocks = (n + threads - 1) / threads;

    fused_gelu_f32<<<blocks, threads>>>(
        out.data_ptr<float>(),
        x.data_ptr<float>(),
        n
    );

    return out;
}
"""

cpp_source = r"""
torch::Tensor cuda_gelu(torch::Tensor x);
"""

In [ ]:
cuda_module = load_inline(
    name="custom_gelu_cuda",
    cpp_sources=cpp_source,
    cuda_sources=cuda_source,
    functions=["cuda_gelu"],
    with_cuda=True,
    extra_cflags=["-O3"],
    extra_cuda_cflags=["-O3"],
    verbose=False
)

In [ ]:
class CudaGELU(nn.Module):
    def forward(self, x):
        return cuda_module.cuda_gelu(x)

cuda_gelu = CudaGELU().cuda()

In [8]:
x = torch.tensor(
    [1.0, -1.0, 0.0, 2.0],
    device="cuda",
    dtype=torch.float32
)

print(cuda_gelu(x))

tensor([ 0.8412, -0.1588,  0.0000,  1.9546], device='cuda:0')


In [ ]:
def benchmark_cuda_gpu(n, repeats=100):
    x = torch.empty(
        n,
        device="cuda",
        dtype=torch.float32
    ).uniform_(-3, 3)

    # Warm-up
    for _ in range(10):
        _ = cuda_gelu(x)

    torch.cuda.synchronize()

    start = time.perf_counter()

    for _ in range(repeats):
        y = cuda_gelu(x)

    torch.cuda.synchronize()

    end = time.perf_counter()

    return (end - start) / repeats

In [9]:
sizes = [
    32_000,
    256_000,
    1_000_000,
    8_000_000
]

for n in sizes:
    t = benchmark_cuda_gpu(n)

    print(
        f"{n:,} elements -> "
        f"{t:.9f} seconds"
    )

32,000 elements -> 0.000019561 seconds
256,000 elements -> 0.000010460 seconds
1,000,000 elements -> 0.000042094 seconds
8,000,000 elements -> 0.000304250 seconds


---

## Level 6 — Triton Kernel

Triton offers a Python-like language for custom GPU kernels. The workload is divided into blocks of elements that are processed in parallel, while Triton handles more of the low-level GPU code generation than native CUDA C++.


In [ ]:
import triton
import triton.language as tl

@triton.jit
def gelu_triton_kernel(
    x_ptr,
    out_ptr,
    n_elements,
    BLOCK_SIZE: tl.constexpr
):
    pid = tl.program_id(0)

    offsets = (
        pid * BLOCK_SIZE
        + tl.arange(0, BLOCK_SIZE)
    )

    mask = offsets < n_elements

    x = tl.load(
        x_ptr + offsets,
        mask=mask
    )

    cube = x * x * x

    inner = (
        tl.sqrt(2.0 / 3.14159265358979)
        * (x + 0.044715 * cube)
    )

    result = (
        0.5
        * x
        * (
            1.0
            + tl.extra.cuda.libdevice.tanh(inner)
        )
    )

    tl.store(
        out_ptr + offsets,
        result,
        mask=mask
    )

In [ ]:
class TritonGELU(torch.nn.Module):

    def forward(self, x):

        out = torch.empty_like(x)

        n = x.numel()

        grid = lambda meta: (
            triton.cdiv(
                n,
                meta["BLOCK_SIZE"]
            ),
        )

        gelu_triton_kernel[grid](
            x,
            out,
            n,
            BLOCK_SIZE=1024
        )

        return out


triton_gelu = TritonGELU().cuda()

In [10]:
x = torch.tensor(
    [1.0, -1.0, 0.0, 2.0],
    device="cuda",
    dtype=torch.float32
)

print(triton_gelu(x))

tensor([ 0.8412, -0.1588,  0.0000,  1.9546], device='cuda:0')


In [ ]:
def benchmark_triton_gpu(n, repeats=100):

    x = torch.empty(
        n,
        device="cuda",
        dtype=torch.float32
    ).uniform_(-3, 3)

    # Warm-up
    for _ in range(10):
        _ = triton_gelu(x)

    torch.cuda.synchronize()

    start = time.perf_counter()

    for _ in range(repeats):
        y = triton_gelu(x)

    torch.cuda.synchronize()

    end = time.perf_counter()

    return (end - start) / repeats


In [11]:
sizes = [
    32_000,
    256_000,
    1_000_000,
    8_000_000
]

for n in sizes:

    t = benchmark_triton_gpu(n)

    print(
        f"{n:,} elements -> "
        f"{t:.9f} seconds"
    )

32,000 elements -> 0.000039445 seconds
256,000 elements -> 0.000026413 seconds
1,000,000 elements -> 0.000036196 seconds
8,000,000 elements -> 0.000269919 seconds


---

## Correctness Check

Performance comparisons are meaningful only if the implementations compute the intended function. The custom PyTorch, compiled, CUDA, and Triton implementations are compared against PyTorch GELU using the **same tanh approximation**.


In [12]:
inputs = torch.tensor(
    [1.0, -1.0, 0.0, 2.0],
    device="cuda",
    dtype=torch.float32,
)

ref = gelu_builtin_tanh(inputs)

def get_err(out):
    return (out - ref).abs().max().item()

print("Correctness check — same GELU approximation:")
print(f"Custom PyTorch: {get_err(gelu_torch(inputs)):.2e}")
print(f"torch.compile:  {get_err(compiled_gelu(inputs)):.2e}")
print(f"CUDA C++:       {get_err(cuda_gelu(inputs)):.2e}")
print(f"Triton:         {get_err(triton_gelu(inputs)):.2e}")


Correctness check — same GELU approximation:
Custom PyTorch: 0.00e+00
torch.compile:  0.00e+00
CUDA C++:       0.00e+00
Triton:         0.00e+00


---

## Final GPU Benchmark

The final benchmark uses GPU warm-up, `torch.cuda.synchronize()`, CUDA events, repeated measurements, and the **median** execution time. Input tensors are already resident on the GPU, so host-to-device transfer time is excluded.

The recorded results below were produced on a Tesla T4. The built-in row uses default `F.gelu(x)` as noted earlier.


In [ ]:
def benchmark(fn, x, warmup=20, rep=100, label=""):
    # Warm-up
    for _ in range(warmup):
        fn(x)

    torch.cuda.synchronize()

    start_events = [
        torch.cuda.Event(enable_timing=True)
        for _ in range(rep)
    ]

    end_events = [
        torch.cuda.Event(enable_timing=True)
        for _ in range(rep)
    ]

    # Benchmark
    for i in range(rep):
        start_events[i].record()

        fn(x)

        end_events[i].record()

    torch.cuda.synchronize()

    times = [
        start.elapsed_time(end)
        for start, end in zip(start_events, end_events)
    ]

    median_ms = sorted(times)[rep // 2]
    mean_ms = sum(times) / len(times)
    min_ms = min(times)

    print(
        f"{label:<25s}"
        f" median: {median_ms:8.4f} ms"
        f" | mean: {mean_ms:8.4f} ms"
        f" | min: {min_ms:8.4f} ms"
    )

    return median_ms

In [13]:
sizes = [
    ("32K",  32 * 1024),
    ("256K", 256 * 1024),
    ("1M",   1024 * 1024),
    ("8M",   8 * 1024 * 1024),
    ("32M",  32 * 1024 * 1024),
]

impl_names = [
    "Custom PyTorch",
    "torch.compile",
    "PyTorch built-in",
    "CUDA C++",
    "Triton",
]

impl_fns = [
    gelu_torch,
    compiled_gelu,
    gelu_builtin,
    cuda_gelu,
    triton_gelu,
]

results = {}

for size_label, n in sizes:
    x = torch.randn(n, device="cuda", dtype=torch.float32)
    print(f"\n── Tensor size: {size_label} ({n:,} elements) ──")

    results[size_label] = {}
    for name, fn in zip(impl_names, impl_fns):
        t = benchmark(fn, x, label=name)
        results[size_label][name] = t



── Tensor size: 32K (32,768 elements) ──
Custom PyTorch            median:   0.1085 ms | mean:   0.1517 ms | min:   0.0989 ms
torch.compile             median:   0.0914 ms | mean:   0.1020 ms | min:   0.0819 ms
PyTorch built-in          median:   0.0206 ms | mean:   0.0237 ms | min:   0.0186 ms
CUDA C++                  median:   0.0207 ms | mean:   0.0215 ms | min:   0.0194 ms
Triton                    median:   0.0440 ms | mean:   0.0462 ms | min:   0.0414 ms

── Tensor size: 256K (262,144 elements) ──
Custom PyTorch            median:   0.1024 ms | mean:   0.1058 ms | min:   0.0979 ms
torch.compile             median:   0.0903 ms | mean:   0.1160 ms | min:   0.0804 ms
PyTorch built-in          median:   0.0205 ms | mean:   0.0224 ms | min:   0.0187 ms
CUDA C++                  median:   0.0216 ms | mean:   0.0233 ms | min:   0.0204 ms
Triton                    median:   0.0469 ms | mean:   0.0512 ms | min:   0.0430 ms

── Tensor size: 1M (1,048,576 elements) ──
Custom PyTorch      

### Recorded Median Times

| Tensor Size | Custom PyTorch | `torch.compile` | PyTorch built-in* | CUDA C++ | Triton |
|---|---:|---:|---:|---:|---:|
| 32K | 0.1085 ms | 0.0914 ms | **0.0206 ms** | 0.0207 ms | 0.0440 ms |
| 256K | 0.1024 ms | 0.0903 ms | **0.0205 ms** | 0.0216 ms | 0.0469 ms |
| 1M | 0.3329 ms | 0.0939 ms | **0.0403 ms** | 0.0465 ms | 0.0529 ms |
| 8M | 2.5660 ms | 0.2793 ms | 0.2906 ms | **0.2725 ms** | 0.2888 ms |
| 32M | 10.2689 ms | 1.1032 ms | 1.1448 ms | **1.0725 ms** | 1.1424 ms |

\* Recorded with default `F.gelu(x)`. For a strictly identical mathematical comparison, rerun the matched-mode cell below.


### Optional Strict Matched-Mode Rerun

This cell replaces the built-in function with `approximate="tanh"` so every GPU implementation uses the same GELU approximation. It is intentionally left unexecuted in this cleaned copy because the original recorded timings were generated with default `F.gelu(x)`.


In [ ]:
matched_impl_names = [
    "Custom PyTorch",
    "torch.compile",
    "PyTorch built-in (tanh)",
    "CUDA C++",
    "Triton",
]

matched_impl_fns = [
    gelu_torch,
    compiled_gelu,
    gelu_builtin_tanh,
    cuda_gelu,
    triton_gelu,
]

matched_results = {}

for size_label, n in sizes:
    x = torch.randn(n, device="cuda", dtype=torch.float32)
    print(f"\n── Tensor size: {size_label} ({n:,} elements) ──")
    matched_results[size_label] = {}

    for name, fn in zip(matched_impl_names, matched_impl_fns):
        t = benchmark(fn, x, label=name)
        matched_results[size_label][name] = t


---

## CPU vs GPU Baseline

This comparison uses the pure Python CPU loop on approximately 8 million elements and the fastest recorded GPU median at the same scale (CUDA C++ at 0.2725 ms). It is an **implementation-level** comparison, not a universal statement about CPU-vs-GPU hardware speed.


In [14]:
import random
import time

n = 8 * 1024 * 1024

random.seed(42)
x_cpu = [random.uniform(-3, 3) for _ in range(n)]

start = time.perf_counter()

y_cpu = [gelu_python(v) for v in x_cpu]

end = time.perf_counter()

cpu_time_s = end - start

print(f"Python CPU: {cpu_time_s:.6f} seconds")

Python CPU: 3.541025 seconds


In [15]:
gpu_time_s = 0.2725 / 1000

speedup = cpu_time_s / gpu_time_s

print(f"CPU time: {cpu_time_s:.6f} s")
print(f"GPU time: {gpu_time_s:.9f} s")
print(f"Speedup: {speedup:,.1f}x")

CPU time: 3.541025 s
GPU time: 0.000272500 s
Speedup: 12,994.6x


---

## Profiler — Eager PyTorch vs `torch.compile`

The benchmark showed a large speedup from compilation. The profiler explains the mechanism: eager PyTorch launches multiple element-wise kernels, while the compiled graph is fused into a Triton-generated kernel.


In [ ]:
from torch.profiler import profile, ProfilerActivity

x = torch.randn(
    8 * 1024 * 1024,
    device="cuda",
    dtype=torch.float32
)

# Warm-up compiled version first
for _ in range(10):
    compiled_gelu(x)

torch.cuda.synchronize()

In [16]:
with profile(
    activities=[
        ProfilerActivity.CPU,
        ProfilerActivity.CUDA
    ]
) as prof_eager:

    for _ in range(10):
        gelu_torch(x)

    torch.cuda.synchronize()

print("CUSTOM PYTORCH")
print(
    prof_eager.key_averages().table(
        sort_by="self_cuda_time_total",
        row_limit=20
    )
)

/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(


CUSTOM PYTORCH
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                              aten::mul         1.99%     532.400us        10.75%       2.884ms      72.088us      12.582ms        50.00%      12.853ms     321.324us            40  
void at::native::vectorized_elementwise_kernel<4, at...         0.00%       0.000us         0.00%       0.000us       0.000us       8.458ms        33.61%       8.458ms     281.931us           

In [17]:
with profile(
    activities=[
        ProfilerActivity.CPU,
        ProfilerActivity.CUDA
    ]
) as prof_compiled:

    for _ in range(10):
        compiled_gelu(x)

    torch.cuda.synchronize()

print("TORCH.COMPILE")
print(
    prof_compiled.key_averages().table(
        sort_by="self_cuda_time_total",
        row_limit=20
    )
)

TORCH.COMPILE
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                    triton_poi_fused_add_mul_pow_tanh_0         3.32%     182.989us         6.26%     345.416us      34.542us       2.478ms       100.00%       2.478ms     247.803us            10  
                    triton_poi_fused_add_mul_pow_tanh_0         0.00%       0.000us         0.00%       0.000us       0.000us       2.478ms       100.00%       2.478ms     275.336us            

---

## Observations

- GPU execution alone does not guarantee optimal performance.
- The eager custom PyTorch implementation launches multiple element-wise kernels.
- `torch.compile` substantially reduced execution time by fusing the computation into a compiled kernel.
- PyTorch's built-in GELU was extremely competitive for small and medium workloads.
- The simple custom CUDA kernel achieved the lowest **recorded** median time at 8M and 32M elements, although the gap among optimized implementations was small.
- Triton approached the performance of optimized CUDA/PyTorch implementations while using a higher-level GPU programming model.
- Performance rankings depend on workload size and implementation details.
- Correctness and benchmark methodology must be verified before interpreting small timing differences.
- The CPU-vs-GPU speedup here reflects both hardware parallelism and a dramatic change in execution strategy.

### Main takeaway

> GPU performance is a systems problem: hardware capability, parallelism, kernel organization, compiler optimization, memory traffic, and workload size all interact to determine execution time.
